# Consolidation: Hand-Built TM vs HARK's Built-In Pipeline

This notebook bridges our hand-built transition matrix prototypes to HARK's
production `NewKeynesianConsumerType` infrastructure.

**Goals:**

1. **Validate** that our hand-built TM matches HARK's `gen_tran_matrix_1D`
   and `calc_transition_matrix()` element-by-element
2. **Map** the built-in API (`define_distribution_grid`, `calc_transition_matrix`,
   `calc_ergodic_dist`, `compute_pe_steady_state`) to the steps we've been
   doing manually
3. **Generalize** — show how the Markov extension adds block structure on top
   of the single-state TM

---

In [ ]:
import time
import numpy as np

import matplotlib.pyplot as plt
import scipy.sparse.linalg as sp_linalg

from HARK.ConsumptionSaving.ConsNewKeynesianModel import (
    NewKeynesianConsumerType,
    init_newkeynesian,
)
from HARK.Calibration.Income.IncomeProcesses import (
    construct_lognormal_income_process_unemployment,
)
from HARK.utilities import (
    jump_to_grid_1D,
    gen_tran_matrix_1D,
)

## Part 1: The Built-In Pipeline

`NewKeynesianConsumerType` provides a complete TM workflow via
`compute_pe_steady_state()`, which internally calls:

1. `solve()` — solve the consumption-saving problem
2. Set `neutral_measure = True` and reconstruct income shocks
3. `define_distribution_grid()` — create the $m$-grid (and $p$-grid $= [1]$ under neutral measure)
4. `calc_transition_matrix()` — build the transition matrix using `gen_tran_matrix_1D`
5. `calc_ergodic_dist()` — find the eigenvector for eigenvalue 1
6. Compute $A_{ss}$ and $C_{ss}$ from the ergodic distribution and policy grids

In [ ]:
nk = NewKeynesianConsumerType(**init_newkeynesian)
A_ss, C_ss = nk.compute_pe_steady_state()

print("Built-in steady state:")
print(f"  A_ss = {A_ss:.8f}")
print(f"  C_ss = {C_ss:.8f}")
print(
    f"  m-grid: {len(nk.dist_mGrid)} points, range [{nk.dist_mGrid[0]:.4f}, {nk.dist_mGrid[-1]:.1f}]"
)
print(f"  p-grid: {nk.dist_pGrid}  (neutral measure → collapsed to [1])")
print(f"  TM shape: {nk.tran_matrix.shape}")
print(f"  Ergodic dist shape: {nk.vec_erg_dstn.shape}")

## Part 2: Replicating by Hand

We'll now do the exact same computation step-by-step, using only
low-level primitives — the same approach from our Markov prototypes.

### Step 2a: Same income distribution (neutral measure)

In [ ]:
neutral_dstn_list = construct_lognormal_income_process_unemployment(
    T_cycle=1,
    PermShkStd=nk.PermShkStd,
    PermShkCount=nk.PermShkCount,
    TranShkStd=nk.TranShkStd,
    TranShkCount=nk.TranShkCount,
    T_retire=0,
    UnempPrb=nk.UnempPrb,
    IncUnemp=nk.IncUnemp,
    UnempPrbRet=None,
    IncUnempRet=None,
    RNG=np.random.default_rng(0),
    neutral_measure=True,
)
neutral_dstn = neutral_dstn_list[0]

shk_prbs = neutral_dstn.pmv
perm_shks = neutral_dstn.atoms[0]
tran_shks = neutral_dstn.atoms[1]

print(f"Shock points: {len(shk_prbs)}")
print(f"Sum of pmv: {shk_prbs.sum():.8f}")
print(f"E*[1/ψ] = {np.sum(shk_prbs / perm_shks):.8f}  (should ≈ 1.0)")

### Step 2b: Same grid and policy functions

In [ ]:
dist_mGrid = nk.dist_mGrid
M = len(dist_mGrid)

cFunc = nk.solution[0].cFunc
cPol = cFunc(dist_mGrid)
aPol = dist_mGrid - cPol
bNext = nk.Rfree[0] * aPol
LivPrb = nk.LivPrb[0]

print("Policy grids match built-in?")
print(f"  cPol: {np.allclose(cPol, nk.cPol_Grid)}")
print(f"  aPol: {np.allclose(aPol, nk.aPol_Grid)}")

### Step 2c: Build TM by hand — Python loop

This is the core of our prototype approach: loop over each grid point $i$,
compute $m'$ for each shock realization, and use `jump_to_grid_1D` to
distribute probability mass onto the grid.

The formula (same as `gen_tran_matrix_1D`):

$$m'_i = \frac{R \cdot a_i}{\psi} + \theta$$

where $(\psi, \theta)$ are drawn with neutral-measure probabilities.

In [ ]:
start = time.time()

NewBornDist = jump_to_grid_1D(np.ones_like(tran_shks), shk_prbs, dist_mGrid)

TM_hand = np.zeros((M, M))
for i in range(M):
    mNext_shks = bNext[i] / perm_shks + tran_shks
    lottery = jump_to_grid_1D(mNext_shks, shk_prbs, dist_mGrid)
    TM_hand[:, i] = LivPrb * lottery + (1.0 - LivPrb) * NewBornDist

elapsed_hand = time.time() - start
print(f"Hand-built TM: {elapsed_hand:.3f} seconds")
print(
    f"Column sums: min={TM_hand.sum(axis=0).min():.10f}, max={TM_hand.sum(axis=0).max():.10f}"
)

### Step 2d: Build TM using `gen_tran_matrix_1D` (numba-compiled)

This is the exact function that `NewKeynesianConsumerType.calc_transition_matrix()`
calls internally. It's `@numba.njit` compiled for speed.

In [ ]:
start = time.time()
TM_numba = gen_tran_matrix_1D(
    dist_mGrid, bNext, shk_prbs, perm_shks, tran_shks, LivPrb, NewBornDist
)
elapsed_numba = time.time() - start
print(
    f"gen_tran_matrix_1D: {elapsed_numba:.3f} seconds (includes JIT compilation on first call)"
)

start = time.time()
TM_numba = gen_tran_matrix_1D(
    dist_mGrid, bNext, shk_prbs, perm_shks, tran_shks, LivPrb, NewBornDist
)
elapsed_numba2 = time.time() - start
print(f"gen_tran_matrix_1D (warm): {elapsed_numba2:.3f} seconds")
print(f"Speedup: {elapsed_hand / max(elapsed_numba2, 1e-8):.0f}×")

## Part 3: Element-by-Element Comparison

All three transition matrices should be identical:
1. **Built-in** (`nk.tran_matrix`) — from `calc_transition_matrix()`
2. **Hand loop** (`TM_hand`) — our Python loop
3. **Numba** (`TM_numba`) — calling `gen_tran_matrix_1D` directly

In [ ]:
TM_builtin = nk.tran_matrix

diff_hand_numba = np.abs(TM_hand - TM_numba).max()
print(f"Max |TM_hand - TM_numba|  = {diff_hand_numba:.2e}")

diff_hand_builtin = np.abs(TM_hand - TM_builtin).max()
print(f"Max |TM_hand - TM_builtin| = {diff_hand_builtin:.2e}")

diff_numba_builtin = np.abs(TM_numba - TM_builtin).max()
print(f"Max |TM_numba - TM_builtin| = {diff_numba_builtin:.2e}")

print()
if diff_hand_builtin < 1e-10:
    print("✓ Hand-built TM matches HARK built-in to machine precision!")
else:
    print(f"✗ Discrepancy detected: {diff_hand_builtin:.2e}")
    print("  (May be due to different RNG seeds for income distribution)")

In [ ]:
eigenvalues, eigenvectors = sp_linalg.eigs(TM_hand, k=1, which="LM", v0=np.ones(M))
erg_hand = eigenvectors[:, 0].real
erg_hand = erg_hand / erg_hand.sum()

erg_builtin = nk.vec_erg_dstn.flatten()

diff_erg = np.abs(erg_hand - erg_builtin).max()
print(f"Max |ergodic_hand - ergodic_builtin| = {diff_erg:.2e}")

A_hand = np.dot(aPol, erg_hand)
C_hand = np.dot(cPol, erg_hand)

print("\nAggregates:")
print(f"  {'':20s} {'Built-in':>12s} {'Hand-built':>12s} {'Diff':>12s}")
print(f"  {'A_ss':20s} {A_ss:12.8f} {A_hand:12.8f} {A_ss - A_hand:12.2e}")
print(f"  {'C_ss':20s} {C_ss:12.8f} {C_hand:12.8f} {C_ss - C_hand:12.2e}")

## Part 4: Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(dist_mGrid, erg_builtin, label="Built-in", linewidth=2)
axes[0].plot(dist_mGrid, erg_hand, "--", label="Hand-built", linewidth=1.5, alpha=0.8)
axes[0].set_xlabel("$m$ (normalized market resources)")
axes[0].set_ylabel("Probability")
axes[0].set_title("Ergodic Distribution")
axes[0].set_xlim([0, 15])
axes[0].legend()

im = axes[1].imshow(
    np.abs(TM_hand - TM_builtin), aspect="auto", cmap="Reds", interpolation="nearest"
)
axes[1].set_title(f"|TM_hand - TM_builtin|\nmax = {diff_hand_builtin:.2e}")
axes[1].set_xlabel("Source state (column)")
axes[1].set_ylabel("Target state (row)")
plt.colorbar(im, ax=axes[1])

axes[2].plot(dist_mGrid, cPol, label="$c(m)$", linewidth=2)
axes[2].plot(dist_mGrid, aPol, label="$a(m)$", linewidth=2)
axes[2].plot(dist_mGrid, dist_mGrid, ":", color="gray", alpha=0.4, label="45° line")
axes[2].set_xlabel("$m$")
axes[2].set_ylabel("Policy")
axes[2].set_title("Consumption & Asset Policies")
axes[2].set_xlim([0, 15])
axes[2].legend()

plt.suptitle("Hand-Built vs HARK Built-In: Exact Match", fontsize=14)
plt.tight_layout()
plt.show()

## Part 5: Anatomy of the Built-In Pipeline

| Step | Hand-built (our prototypes) | HARK built-in |
|------|---------------------------|---------------|
| Grid | `make_grid_exp_mult(...)` | `nk.define_distribution_grid()` → `nk.dist_mGrid` |
| Policy | `cFunc(dist_mGrid)` | `nk.calc_transition_matrix()` → `nk.cPol_Grid`, `nk.aPol_Grid` |
| Newborns | `jump_to_grid_1D(...)` | Built into `calc_transition_matrix()` |
| TM build | Python loop with `jump_to_grid_1D` | `gen_tran_matrix_1D()` (numba) → `nk.tran_matrix` |
| Ergodic | `scipy.sparse.linalg.eigs(...)` | `nk.calc_ergodic_dist()` → `nk.vec_erg_dstn` |
| Aggregates | `np.dot(cPol, erg_dist)` | `nk.A_ss`, `nk.C_ss` |
| All-in-one | — | `nk.compute_pe_steady_state()` |

The built-in pipeline adds:
- **Numba JIT compilation** for the inner loop (`gen_tran_matrix_1D`)
- **Automatic grid construction** based on `mMin`, `mMax`, `mCount`, `mFac`
- **Neutral measure toggle** via `nk.neutral_measure = True`
- **Jacobian computation** via `nk.calc_jacobian()` for sequence-space methods

## Part 6: The Markov Extension as Block Structure

Our Markov TM prototypes are a direct generalization of the single-state TM.

For $J$ Markov states, the full TM has size $(M \times J)^2$ with block structure:

$$\mathbf{T} = \begin{pmatrix}
T_{0\to 0} & T_{1\to 0} & \cdots & T_{J-1\to 0} \\
T_{0\to 1} & T_{1\to 1} & \cdots & T_{J-1\to 1} \\
\vdots & \vdots & \ddots & \vdots \\
T_{0\to J-1} & T_{1\to J-1} & \cdots & T_{J-1\to J-1}
\end{pmatrix}$$

where each $M \times M$ block $T_{j \to j'}$ satisfies:

$$T_{j \to j'}[:, i] = \pi_{j \to j'} \cdot \lambda_j \cdot \text{jump\_to\_grid\_1D}(m'_i, \text{probs}, \text{mGrid}) + (1-\lambda_j) \cdot \text{NewBorn}_{j'}$$

with $m'_i = R_{j'} \cdot a_j(m_i) / (\psi \cdot \Gamma_{j'}) + \theta$.

When $J=1$, this reduces exactly to the single-state TM above.

### Demonstration: Single Markov state = Single-state TM

We verify that our Markov TM code with $J=1$ produces the same matrix as
the built-in pipeline.

In [ ]:
J = 1
MrkvArr = np.array([[1.0]])
Rfree_arr = np.array([nk.Rfree[0]])
LivPrb_arr = np.array([nk.LivPrb[0]])
PermGroFac_arr = np.array([nk.PermGroFac[0]])

cPol_list = [cPol]
aPol_list = [aPol]
N_states = M * J

NewBornDist_markov = np.zeros(N_states)
newborn_1d = jump_to_grid_1D(np.ones_like(tran_shks), shk_prbs, dist_mGrid)
NewBornDist_markov[0:M] = 1.0 * newborn_1d

TM_markov = np.zeros((N_states, N_states))

for j in range(J):
    for jp in range(J):
        markov_prob = MrkvArr[j, jp]
        Rfree_jp = Rfree_arr[jp]
        PermGroFac_jp = PermGroFac_arr[jp]

        for i in range(M):
            bNext_i = Rfree_jp * aPol_list[j][i]
            mNext_shks = bNext_i / (perm_shks * PermGroFac_jp) + tran_shks
            lottery = jump_to_grid_1D(mNext_shks, shk_prbs, dist_mGrid)

            src_idx = j * M + i
            TM_markov[jp * M : (jp + 1) * M, src_idx] += (
                markov_prob * LivPrb_arr[j] * lottery
            )

    for i in range(M):
        src_idx = j * M + i
        TM_markov[:, src_idx] += (1.0 - LivPrb_arr[j]) * NewBornDist_markov

diff_markov = np.abs(TM_markov - TM_builtin).max()
print(f"Max |TM_markov(J=1) - TM_builtin| = {diff_markov:.2e}")

if diff_markov < 1e-10:
    print("✓ Markov code with J=1 exactly reproduces the built-in TM!")
else:
    print(f"✗ Discrepancy: {diff_markov:.2e}")

## Part 7: Monte Carlo Validation

As a final check, compare the TM aggregates with a Monte Carlo simulation
of the same model.

**Important subtlety:** Under the neutral measure, the TM computes
$C_{ss} = E^*[c(m)] = E[c(m) \cdot p] / E[p]$, which is the
*level-weighted* normalized aggregate — NOT the plain cross-sectional
mean $E[c(m)]$.  These differ because $\text{cov}(c(m), p) \neq 0$:
the same permanent shock $\psi$ that increases $p$ also *decreases*
normalized resources $m$ (and hence $c$).

The correct MC comparison is: $C_{ss} \approx \text{mean}(c_i \cdot p_i) / \text{mean}(p_i)$.

In [ ]:
nk_mc = NewKeynesianConsumerType(**init_newkeynesian)
nk_mc.assign_parameters(AgentCount=50000, T_sim=1200)
nk_mc.solve()

nk_mc.track_vars = ["aNrm", "mNrm", "pLvl"]
nk_mc.initialize_sim()
nk_mc.simulate()

mc_cNrm = nk_mc.state_now["mNrm"] - nk_mc.state_now["aNrm"]
mc_pLvl = nk_mc.state_now["pLvl"]

MC_C_nrm = np.mean(mc_cNrm)
MC_A_nrm = np.mean(nk_mc.state_now["aNrm"])
MC_C_lvl = np.mean(mc_cNrm * mc_pLvl)
MC_A_lvl = np.mean(nk_mc.state_now["aNrm"] * mc_pLvl)
MC_MeanP = np.mean(mc_pLvl)

MC_C_neutral = MC_C_lvl / MC_MeanP
MC_A_neutral = MC_A_lvl / MC_MeanP

print(f"MC Mean pLvl = {MC_MeanP:.6f}  (should ≈ 1.0 when PermGroFac=1.0)")
print()
print("=== Correct comparison: TM (neutral measure) vs MC level aggregates / MeanP ===")
print(f"{'':20s} {'TM E*[·]':>12s} {'MC E[·p]/p̄':>12s} {'Diff':>12s} {'Pct':>8s}")
diff_c = C_ss - MC_C_neutral
diff_a = A_ss - MC_A_neutral
pct_c = 100 * diff_c / C_ss if C_ss != 0 else 0
pct_a = 100 * diff_a / A_ss if A_ss != 0 else 0
print(
    f"{'Consumption':20s} {C_ss:12.6f} {MC_C_neutral:12.6f} {diff_c:12.6f} {pct_c:7.2f}%"
)
print(f"{'Assets':20s} {A_ss:12.6f} {MC_A_neutral:12.6f} {diff_a:12.6f} {pct_a:7.2f}%")
print()
print("=== Why E[c] ≠ E*[c]: the neutral measure reweights by p ===")
print(f"  MC E[cNrm]       = {MC_C_nrm:.6f}  (plain normalized mean)")
print(f"  MC E[cNrm·p]/p̄  = {MC_C_neutral:.6f}  (level-weighted, what TM computes)")
print(f"  TM C_ss          = {C_ss:.6f}")
print()
print(f"  MC E[aNrm]       = {MC_A_nrm:.6f}")
print(f"  MC E[aNrm·p]/p̄  = {MC_A_neutral:.6f}")
print(f"  TM A_ss          = {A_ss:.6f}")
print()
print("  E[c] ≠ E*[c] because cov(c(m), p) < 0: agents with high p")
print("  tend to have lower normalized m (same ψ shock decreases m and increases p).")

## Summary

**Key findings:**

1. **Exact numerical match:** The hand-built TM, `gen_tran_matrix_1D`, and
   `calc_transition_matrix()` all produce identical matrices (to machine precision).

2. **Same aggregates:** The ergodic distributions and steady-state aggregates
   ($A_{ss}$, $C_{ss}$) match exactly.

3. **Markov = block generalization:** Our Markov TM code with $J=1$ reduces
   to the built-in single-state TM, confirming that the Markov extension is
   a clean generalization.

4. **Neutral measure subtlety:** $C_{ss} = E^*[c(m)] \neq E[c(m)]$ — the
   neutral-measure aggregate equals the *level-weighted* normalized mean,
   not the plain cross-sectional mean.  This is correct and intentional:
   it gives aggregates in level terms (after multiplying by $\bar{p}$).

5. **Speed:** `gen_tran_matrix_1D` (numba) is faster than the Python loop
   for large grids, but produces identical results.

**Implications for production code:**

The pathway from prototypes to HARK integration is clear:
- Wrap the Markov TM loop in a `gen_tran_matrix_1D_markov` function
- Add `define_distribution_grid()` and `calc_transition_matrix()` methods
  to `MarkovConsumerType`
- Support `neutral_measure=True` for models with varying PermGroFac
- Enable Jacobian computation for Markov-state HANK models

### Notebook progression

| # | Notebook | New concept |
|---|----------|-------------|
| 1 | `markov-tm-prototype` | Hand-built TM for 2-state Markov |
| 2 | `serial-unemployment-tm` | Scaling to 4 states |
| 3 | `serial-growth-tm-2d` | 2D grid for PermGroFac ≠ 1 |
| 4 | `serial-growth-tm-harmenberg` | Harmenberg neutral measure |
| 5 | **`tm-consolidation`** | **Validates hand-built = HARK built-in** |
| 6 | (next) `agg-shock-markov-tm` | Endogenous aggregate state |